# satellite_topology_viewer_3D 使用说明

这个 notebook 用来记录 3D 拓扑 viewer 的典型用法。每个代码单元都尽量自包含，可以单独运行。

设计原则：

- `module/` 只放通用能力，不硬编码 Starlink、G60 或某个数据目录。
- `examples/configs/` 放具体星座、缓存路径和 group YAML。
- GUI 只显示已经准备好的数据，不在 viewer 里面做 rev-group、节点偏移或 group 转换。
- 3D 和 2D 都使用内部 0-based node 编号：`node = x * N + y`。
- 现在 3D 选中卫星时显示 `node=962 (x=43, y=16)` 这种 0-based 信息，方便直接和 2D 对齐。

## 1. 一键运行脚本

100s GUI：

```powershell
& 'C:/ProgramData/miniconda3/envs/paper11/python.exe' 'E:/paper11/generic/src/satellite_topology_viewer_3D/examples/run_starlink_2d3d_viewer.py'
```

只检查数据装配，不弹 GUI：

```powershell
& 'C:/ProgramData/miniconda3/envs/paper11/python.exe' 'E:/paper11/generic/src/satellite_topology_viewer_3D/examples/run_starlink_2d3d_viewer.py' --check-only --start 1 --end 20
```

86100s GUI：

```powershell
& 'C:/ProgramData/miniconda3/envs/paper11/python.exe' 'E:/paper11/generic/src/satellite_topology_viewer_3D/examples/run_starlink_2d3d_viewer.py' --config 'E:/paper11/generic/src/satellite_topology_viewer_3D/examples/configs/starlink_2d3d_viewer_full.yaml'
```

## 2. 通过 YAML 装配 2D + 3D GUI

这个单元适合平时直接看 100s 的 2D+3D 同步效果。Jupyter 里需要启用 Qt 事件循环，代码里已经包含 `get_ipython().run_line_magic("gui", "qt5")`。

In [ ]:
import sys
from pathlib import Path

# Jupyter/Notebook 中启用 Qt 事件循环；等价于单独运行：%gui qt5
try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path("E:/paper11/generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.satellite_topology_viewer_3D.module import (
    Synced2D3DTopologyWindow,
    load_synced_2d3d_inputs_from_yaml,
)

CONFIG_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "starlink_2d3d_viewer_test100.yaml"
START = 0
END = 100
STRIDE = 1

inputs = load_synced_2d3d_inputs_from_yaml(
    CONFIG_FILE,
    start_override=START,
    end_override=END,
    stride_override=STRIDE,
)

app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])
viewer = Synced2D3DTopologyWindow(
    config=inputs.config,
    delay_data=inputs.delay_data,
    position_series=inputs.position_series,
    group_data=inputs.group_data,
    show_groups=inputs.groups_enabled,
    show_3d_links=True,
    show_3d_orbits=True,
    link_stride=1,
    timer_interval_ms=180,
)
viewer.resize(1600, 900)
viewer.show()

# 保留引用，避免 notebook 单元结束后窗口对象被回收。
_viewer_refs = globals().setdefault("_viewer_refs", [])
_viewer_refs.append(viewer)

print(
    f"shown | config={inputs.config.name} | steps={len(inputs.delay_data.steps)} | "
    f"edges={inputs.delay_data.edge_table.num_edges} | position_cache={inputs.position_cache_dir}"
)
viewer

## 3. 显式指定路径，不走总控 YAML

如果你想在别的脚本或 notebook 里使用模块，可以直接指定 edge delay store、position cache、group XML 和 group scheme。这个单元不依赖上一单元的变量。

In [ ]:
import sys
from pathlib import Path

try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path("E:/paper11/generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.satellite_topology_viewer.module import load_edge_delay_data_for_viewer, load_or_build_group_data
from src.satellite_topology_viewer_3D.module import (
    Synced2D3DTopologyWindow,
    load_position_series,
    load_viewer_config,
)

VIEWER_CONFIG_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "viewer_config_starlink_72_22.yaml"
GROUP_SCHEME_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "station_groups_default.yaml"
STORE_DIR = Path("E:/paper11/data/basic_file/Starlink_72_22_1_550/satellitesposition/full_option_edge_delay_test/Starlink_72_22_full_options_t0_100_stride1")
POSITION_CACHE_DIR = Path("E:/paper11/data/basic_file/Starlink_72_22_1_550/satellitesposition/_position_cache/cache_0_100_1s")
GROUP_XML = Path("E:/paper11/data/basic_file/Starlink_72_22_1_550/satellitesposition/station_visible_satellites_72_22_1_delta.xml")
GROUP_CACHE_DIR = Path("E:/paper11/data/basic_file/Starlink_72_22_1_550/satellitesposition/full_option_edge_delay_test/group_data_cache")
EDGE_OPTIONS = (0, 1, 2, 4)
START = 0
END = 100
STRIDE = 1

config = load_viewer_config(VIEWER_CONFIG_FILE, GROUP_SCHEME_FILE)
delay_data = load_edge_delay_data_for_viewer(
    store_dir=STORE_DIR,
    config=config,
    start=START,
    end=END,
    stride=STRIDE,
    options=EDGE_OPTIONS,
)
position_series = load_position_series(
    full_cache_dir=POSITION_CACHE_DIR,
    start=START,
    end=END,
    stride=STRIDE,
)
group_data = load_or_build_group_data(
    xml_file=GROUP_XML,
    group_cache_dir=GROUP_CACHE_DIR,
    steps=delay_data.steps,
    station_groups=config.station_groups,
    total_sats=config.total_sats,
    constellation_name=config.name,
    stride=STRIDE,
    enabled=True,
    force=False,
)

assert delay_data.steps == position_series.steps

app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])
viewer = Synced2D3DTopologyWindow(
    config=config,
    delay_data=delay_data,
    position_series=position_series,
    group_data=group_data,
    show_groups=True,
    show_3d_links=True,
    show_3d_orbits=True,
    link_stride=1,
    timer_interval_ms=180,
)
viewer.resize(1600, 900)
viewer.show()

_viewer_refs = globals().setdefault("_viewer_refs", [])
_viewer_refs.append(viewer)

print(f"shown | steps={len(delay_data.steps)} | edges={delay_data.edge_table.num_edges}")
viewer

## 4. 只使用 3D 父类

`SatelliteGlobe3DWidget` 是 3D 父类，只负责 3D 显示。它不依赖 2D viewer。适合单独检查 position cache、轨道线和链路。

In [ ]:
import sys
from pathlib import Path

try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path("E:/paper11/generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.link_delay.module.edge_options import build_full_option_edges
from src.satellite_topology_viewer_3D.module import (
    SatelliteGlobe3DWidget,
    load_position_series,
    load_viewer_config,
)

VIEWER_CONFIG_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "viewer_config_starlink_72_22.yaml"
GROUP_SCHEME_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "station_groups_default.yaml"
POSITION_CACHE_DIR = Path("E:/paper11/data/basic_file/Starlink_72_22_1_550/satellitesposition/_position_cache/cache_0_100_1s")
START = 0
END = 100
STRIDE = 1

config = load_viewer_config(VIEWER_CONFIG_FILE, GROUP_SCHEME_FILE)
position_series = load_position_series(
    full_cache_dir=POSITION_CACHE_DIR,
    start=START,
    end=END,
    stride=STRIDE,
)
edge_table = build_full_option_edges(config, options=(0, 1, 2, 4))

app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])
viewer3d = SatelliteGlobe3DWidget(
    config,
    position_series=position_series,
    edge_table=edge_table,
    edge_values=None,
    group_data=None,
    show_groups=False,
    show_links=True,
    show_orbits=True,
    link_stride=1,
)
viewer3d.resize(900, 760)
viewer3d.show()

_viewer_refs = globals().setdefault("_viewer_refs", [])
_viewer_refs.append(viewer3d)

print(f"3D only | steps={position_series.num_steps} | sats={position_series.num_sats} | edges={edge_table.num_edges}")
viewer3d

## 5. 检查某个时间片的 node 和 group

这个单元用于人工核对编号。注意 3D 现在显示的是 0-based node。比如 `node=962` 对应 `x=43, y=16`，因为 `divmod(962, 22) = (43, 16)`。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path("E:/paper11/generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.satellite_topology_viewer_3D.module import load_synced_2d3d_inputs_from_yaml

CONFIG_FILE = GENERIC_ROOT / "src" / "satellite_topology_viewer_3D" / "examples" / "configs" / "starlink_2d3d_viewer_full.yaml"
STEP = 20672
NODE = 962

inputs = load_synced_2d3d_inputs_from_yaml(
    CONFIG_FILE,
    start_override=STEP,
    end_override=STEP,
)

x, y = divmod(NODE, inputs.config.N)
groups_at_step = inputs.group_data.get(STEP, {}).get("groups", {})
memberships = []
for gid, nodes in groups_at_step.items():
    if NODE in set(int(n) for n in nodes):
        name = inputs.config.station_groups.get(int(gid), {}).get("name", f"Group {gid}")
        memberships.append((int(gid), name))

print(f"step={STEP}, node={NODE}, x={x}, y={y}, groups={memberships or 'none'}")